In [4]:
from typing import Any, List, Optional, Callable, Sequence, Tuple, Union
from jaxtyping import Array, Float, Int, PyTree # https://github.com/google/jaxtyping
from jax.random import PRNGKey

import dataclasses

import jax
import jax.numpy as jnp
import numpy as np
import math
import distrax
print("Distrax version:", distrax.__version__)
import flax
from flax import nnx
print("Flax version:", flax.__version__)

Distrax version: 0.1.8
Flax version: 0.12.7


In [19]:
from distrax._src.bijectors.masked_coupling import MaskedCoupling
from distrax._src.bijectors.inverse import Inverse
from distrax._src.bijectors.chain import Chain
from distrax._src.distributions.transformed import Transformed

class MLP(nnx.Module):
    def __init__(self,
                 layer_dims: Sequence[tuple],
                 activation: Callable = nnx.leaky_relu,
                 use_bias: bool = True,
                 rngs: nnx.Rngs = nnx.Rngs(0),
                 kernel_init: Callable = nnx.initializers.lecun_normal
                 ):
                 
        assert len(layer_dims)>0, "At least one layer dimension must be specified in layer_dims."
        self.activation = activation
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        self.layers = nnx.List()
        for layer_dim in layer_dims:
            self.layers.append(nnx.Linear(layer_dim[0], 
                                          layer_dim[1], 
                                          rngs=rngs, 
                                          use_bias=use_bias, 
                                          kernel_init=kernel_init
                                          )
                                )
    def __call__(self, x):
        for l, layer in enumerate(self.layers[:-1]):
            x = self.activation(layer(x))
        x = self.layers[-1](x)
        return x
        

class Conditioner(nnx.Module):
    def __init__(self,
                 features_shape: tuple,
                 context_shape: tuple,
                 hidden_dims: Sequence[tuple],
                 num_bijector_params: int,
                 activation: Callable = nnx.leaky_relu,
                 rngs: nnx.Rngs = nnx.Rngs(0),
                 kernel_init: Callable = nnx.initializers.lecun_normal(),
                 ):
        self.features_shape = features_shape
        self.context_shape = context_shape
        self.num_bijector_params = num_bijector_params
        self.activation = activation

        self.n_flat_features = jax.tree.reduce(lambda carry, x: carry*x, features_shape)
        self.n_flat_context = jax.tree.reduce(lambda carry, x: carry*x, context_shape)
        self.layer_dims = ((self.n_flat_features+self.n_flat_context, hidden_dims[0][0]),) + hidden_dims

        self._conditioner = nnx.List()
        self._conditioner_mlp = MLP(self.layer_dims, activation, rngs=rngs, kernel_init=kernel_init) # Build the NN as specified by layer_dims that learns the sline transformation parameters
        self._conditioner_out = nnx.Linear(self.layer_dims[-1][-1],
                                            self.n_flat_features*num_bijector_params,
                                            rngs=rngs,
                                            kernel_init=kernel_init
                                            )
    
    def __call__(self, x, context=None):
        # Flatten feature vector to prepare for parsing into spline transform Conditioner, which is a multilayer perceptron
        x_batch_shape = x.shape[:-len(self.features_shape)]
        x.reshape(*x_batch_shape, -1)

        if context is not None:
            # Stack the flattened context vector to the flattened feature vector for the Conditioner in a conditional flow transform
            context_batch_shape = context.shape[:-len(self.context_shape)]
            assert x_batch_shape == context_batch_shape, f"Batch shape mismatch: features (x) has shape {x_batch_shape}, context has shape {context_batch_shape}"
            context.reshape(*context_batch_shape, -1)
            x = jnp.hstack([context,x])

        x = self._conditioner_mlp(x)
        x = self.activation(x)
        x = self._conditioner_out(x)
        x = x.reshape(*x_batch_shape, *(self.features_shape + (self.num_bijector_params,)))
        return x


class ConditionalInverse(Inverse):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x: Array, context: Optional[Array] = None) -> Array:
        return self._bijector.inverse(x, context)

    def inverse(self, y: Array, context: Optional[Array] = None) -> Array:
        return self._bijector.forward(y, context)

    def forward_and_log_det(self, x: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        return self._bijector.inverse_and_log_det(x, context)

    def inverse_and_log_det(self, y: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        return self._bijector.forward_and_log_det(y, context)



class ConditionalMaskedCoupling(MaskedCoupling):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, x: Array, context: Optional[Array] = None) -> Array:
        y, _ = self.forward_and_log_det(x, context=context)
        return y

    def inverse(self, y: Array, context: Optional[Array] = None) -> Array:
        x, _ = self.inverse_and_log_det(y, context=context)
        return x

    def forward_and_log_det(self, x: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        self._check_forward_input_shape(x)
        masked_x = jnp.where(self._event_mask, x, 0.0)
        params = self._conditioner(masked_x, context)
        y0, log_d = self._inner_bijector(params).forward_and_log_det(x)
        y = jnp.where(self._event_mask, x ,y0)
        logdet = math.sum_last(
            jnp.where(self._mask, 0., log_d),
            self._event_ndims - self._inner_event_ndims
        )
        # Or sum log-det Jacobian over event dimensions, robust to N-D feature(event) dimensions
        # event_dims = tuple(range(-self._event_ndims, 0)) if self._event_ndims > 0 else ()
        # logdet = jnp.sum(
        #     jnp.where(self._mask, 0., log_d),
        #     axis=event_dims
        # )
        return y, logdet

    def inverse_and_log_det(self, y: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        self._check_inverse_input_shape(y)
        masked_y = jnp.where(self._event_mask, y, 0.0)
        params = self._conditioner(masked_y, context)
        x0, log_d = self._inner_bijector(params).inverse_and_log_det(y)
        x = jnp.where(self._event_mask, y, x0)
        logdet = math.sum_last(
            jnp.where(self._event_mask, 0., log_d),
            self._event_ndims - self._inner_event_ndims
        )
        return x, logdet


class ConditionalChain(Chain):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def forward(self, x: Array, context: Optional[Array] = None) -> Array:
        for bijector in reversed(self._bijectors):
            x = bijector.forward(x, context)
        return x

    def inverse(self, y: Array, context: Optional[Array] = None) -> Array:
        for bijector in self._bijectors:
            y = bijector.inverse(y, context)
        return y

    def forward_and_log_det(self, x: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        x, log_det = self._bijectors[-1].forward_and_log_det(x, context)
        for bijector in reversed(self._bijectors[:-1]):
            x, ld = bijector.forward_and_log_det(x, context)
            log_det += ld
        return x, log_det

    def inverse_and_log_det(self, y: Array, context: Optional[Array] = None) -> Tuple[Array, Array]:
        y, log_det = self._bijectors[0].inverse_and_log_det(y, context)
        for bijector in self._bijectors[1:]:
            y, ld = bijector.inverse_and_log_det(y, context)
            log_det += ld
        return y, log_det


class ConditionalTransformed(Transformed):
    def __init__(self, distribution, flow):
        super().__init__(distribution, flow)

    def log_prob(self, y: Array, context: Optional[Array] = None) -> Array:
        """See `Distribution.log_prob`."""
        x, ildj_y = self.bijector.inverse_and_log_det(y, context=context)
        lp_x = self.distribution.log_prob(x)
        lp_y = lp_x + ildj_y
        return lp_y

    def sample(self,
                *,
                seed: Union[int, PRNGKey],
                sample_shape: Tuple[int,],
                context: Optional[Array] = None) -> Array:
        """Samples an event.

        Args:
        seed: PRNG key or integer seed.
        sample_shape: Additional leading dimensions for sample.
        context: Array of conditional context vectors.

        Returns:
        A sample of shape `sample_shape + self.batch_shape + self.event_shape`.
        """
        rng, sample_shape = convert_seed_and_sample_shape(seed, sample_shape)
        num_samples = functools.reduce(operator.mul, sample_shape, 1)  # product

        samples = self._sample_n(rng, num_samples, context)
        return jax.tree_util.tree_map(
            lambda t: t.reshape(sample_shape + t.shape[1:]), samples)

    def _sample_n(self, key: PRNGKey, n: int, context: Optional[Array] = None) -> Array:
        """Returns `n` samples. Used by .sample method."""
        x = self.distribution.sample(seed=key, sample_shape=n)
        y = jax.vmap(self.bijector.forward)(x, context)
        return y

    def _sample_n_and_log_prob(self, key: PRNGKey, n: int, context: Optional[Array] = None) -> Tuple[Array, Array]:
        """Returns `n` samples and their log probs.

        This function is more efficient than calling `sample` and `log_prob`
        separately, because it uses only the forward methods of the bijector. It
        also works for bijectors that don't implement inverse methods.

        Args:
        key: PRNG key.
        n: Number of samples to generate.

        Returns:
        A tuple of `n` samples and their log probs.
        """
        x, lp_x = self.distribution.sample_and_log_prob(seed=key, sample_shape=n)
        y, fldj = jax.vmap(self.bijector.forward_and_log_det)(x, context)
        lp_y = jax.vmap(jnp.subtract)(lp_x, fldj)
        return y, lp_y
    

class RQSplineFlow(nnx.Module):
    # Needed to make sure that non-nnx.Module objects from distrax are tracked by nnx
    # Alternative implementation is to use nnx.data(...) in the actual code and call on the 
    # conditioners: nnx.List[Conditioner]
    # transforms: list
    # inverse_bijector: nnx.Data[ConditionalInverse]
    # base_dist: nnx.Data[distrax.Distribution]
    # inverse_flow: nnx.Data[ConditionalTransformed]

    def __init__(self,
                 n_features: int,
                 n_context: int = 0,
                 n_transforms: int = 4,
                 hidden_dims: tuple = ((32,32), (32,32)),
                 activation: str = "gelu",
                 n_bins: int = 8,
                 range_min: float = -1.0,
                 range_max: float = 1.0,
                 bijector_type: Callable = distrax.RationalQuadraticSpline,
                 ):
        
        self.features_shape = (n_features, )
        self.context_shape = (n_context, )
        self.n_bins = n_bins
        self.range_min = range_min
        self.range_max = range_max
        self.num_bijector_params = 3*self.n_bins + 1
        self.bijector_type = bijector_type
        self.n_transforms = n_transforms # Number of conditioner-bijector layers in the overall flow

        # Bijector transformation used in each layer of the flow
        # bijector_fn currently defined only for distrax.RationalQuadraticSpline
        def bijector_fn(params: Array):
            return self.bijector_type(params, self.range_min, self.range_max)
        self.bijector_fn = bijector_fn

        # Instantiate all the conditioners needed for #n_transform RQSpline transforms
        self.hidden_dims = hidden_dims
        self.conditioners = nnx.List() # Use nnx.List such that nnx knows to track the conditioner nnx modules
        for t in range(self.n_transforms):
            self.conditioners.append(
                Conditioner(self.features_shape, self.context_shape, self.hidden_dims, self.num_bijector_params)
            )

        """
        Make distrax distribution containing the rational quadratic spline flow.

        Returns:
            Base Gaussian transformed by rational quadratic spline flow.
        """
        # First create alternating binary mask for the masked coupling mechanism 
        mask = jnp.arange(0, np.prod(self.features_shape)) % 2
        mask = jnp.reshape(mask, self.features_shape)
        mask = mask.astype(bool)

        # Now instantiate all the coupling trandforms
        self.transforms = []
        for t in range(self.n_transforms):
            self.transforms.append(
                ConditionalMaskedCoupling(mask=mask, bijector=self.bijector_fn, conditioner=self.conditioners[t])
                )
            mask = jnp.logical_not(mask) # Flipping the binary mask on the input features to the conditioner for each coupling transform
        self.transforms = nnx.data(self.transforms)

        # Now stack all transforms as single distrax.bjector child class 
        # Note that the convention used here is such that the RQSpline transforms map from data features to the conditional context,
        # We need to invert it for probability evaluation
        # inverse_flow and inverse_bijector refers to the flow transform mapping from: latent sampling distribution space --> feature space
        self.inverse_bijector = nnx.data(
                ConditionalInverse(
                    ConditionalChain(self.transforms)
                )
            )   
        self.base_dist = nnx.data(
                distrax.MultivariateNormalDiag(jnp.zeros(self.features_shape), jnp.ones(self.features_shape))
            )
        self.inverse_flow = nnx.data(
                ConditionalTransformed(self.base_dist, self.inverse_bijector)
            )

    def __call__(self, x: Array, context: Optional[Array] = None) -> Array:
        """
        Evaluate the log probability of the flow for non-batched input x.

        Args:
            x (jnp.ndarray (ndim)): Sample at which to predict posterior value.

        Returns:
            jnp.ndarray (float): Predicted log_e posterior value.
        """
        return self.inverse_flow.log_prob(x, context)

    def sample(self, num_samples: int, rng: Array, context: Array = None) -> Array:
        return self.inverse_flow.sample(seed=rng, sample_shape=(num_samples,), context=context)




In [20]:
n_features = 6
n_context = 6

model = RQSplineFlow(
    n_features=n_features,
    n_context=n_context,
    )
    

In [ ]:
def loss_fn()

def mse_loss(model, x, y):
    preds = model(x)
    if preds.shape != y.shape:
        raise ValueError(f"Output shpae of the model {preds.shape} does not match the shape of the labels {y.shape}")
    return jnp.mean((preds-y) ** 2)